# SinMorphy Lemma Reconstruction

This notebook reconstructs the `century_tokens.json` structure using extracted Lemmas from `century_analysis_results_v4.json`.

In [1]:
import json
import os
import traceback

In [2]:
# File Paths
INPUT_TOKENS_FILE = 'Data/century_tokens.json'
INPUT_ANALYSIS_FILE = 'Data/century_analysis_results_v6.json'
OUTPUT_LEMMAS_FILE = 'Data/century_lemmas.json'

def load_json(path):
    if os.path.exists(path):
        with open(path, 'r', encoding='utf-8') as f:
            return json.load(f)
    return None

# Load Original Tokens
century_tokens = load_json(INPUT_TOKENS_FILE)
if not century_tokens:
    print("Error: century_tokens.json not found!")
else:
    print(f"Loaded tokens for {len(century_tokens)} centuries")

# Load Analysis Results
analysis_results = load_json(INPUT_ANALYSIS_FILE)
if not analysis_results:
    # Fallback if cleaned file not found (e.g. user skipped that step)
    print(f"Warning: {INPUT_ANALYSIS_FILE} not found. Trying V5 output...")
    analysis_results = load_json(INPUT_ANALYSIS_FILE)

if not analysis_results:
    print("Error: No analysis results found!")
else:
    print(f"Loaded analysis results from {INPUT_ANALYSIS_FILE}")


Loaded tokens for 9 centuries
Loaded analysis results from Data/century_analysis_results_v6.json


In [3]:
def reconstruct_data(tokens_dict, results_dict):
    reconstructed = {}
    
    # Iterate through the original keys to preserve structure order
    for century, original_sentences in tokens_dict.items():
        reconstructed[century] = []
        
        result_list = results_dict.get(century, [])
        # Create a lookup: index -> analysis_entry
        result_map = {item['sentence_index']: item for item in result_list}
        
        for i, original_tokens in enumerate(original_sentences):
            lemma_sentence = []
            
            # Retrieve analysis for this sentence index
            analysis_entry = result_map.get(i)
            
            if not analysis_entry:
                # Missing analysis -> Warning + Fallback to original tokens
                # print(f"Warning: Missing analysis for Century {century} Sentence {i}")
                lemma_sentence = original_tokens
            else:
                extracted_analysis = analysis_entry.get('analysis', [])
                
                # Direct list processing
                for item in extracted_analysis:
                    # V5 simplified logic: Just take 'lemma' or 'word'
                    lemma = item.get('lemma')
                    if not lemma or lemma == 'None':
                        lemma = item.get('word')
                    
                    if not lemma: 
                         lemma = ""
                         
                    lemma_sentence.append(lemma)
            
            reconstructed[century].append(lemma_sentence)
            
    return reconstructed

if century_tokens and analysis_results:
    print("Reconstructing data...")
    century_lemmas = reconstruct_data(century_tokens, analysis_results)
    print("Done.")
    
    # Validate Sample
    first_key = list(century_lemmas.keys())[0]
    print(f"Sample from Century {first_key}:")
    print("Original:", century_tokens[first_key][0])
    print("Lemmas:  ", century_lemmas[first_key][0])

Reconstructing data...
Done.
Sample from Century 5:
Original: ['කාම', 'ලොක', 'රූප', 'ලොක', 'අරූපලෝක', 'යන්', 'ලොකත්රයට', 'ශ්රෙෂ්ඨ', 'වූ', 'මුනීන්ද්ර', 'නම්', 'වූ', 'සම්යක්', 'සම්බුද්ධයන්', 'වහන්සේගේ', 'ශ්රී', 'පාද', 'පද්මය', 'භය', 'ලොහ', 'කුලාචාරයෙන්', 'විනා', 'සකසා', 'වැඳ', 'සකල', 'ශාස්ත්රයෙහි', 'පාරප්රාප්ත', 'පණ්ඩිත', 'වූ', 'භෛෂජ්යාර්ථයෙහි', 'දක්ෂ', 'වූ', 'වෙදුන්', 'විසින්', 'උතුම්', 'වූ', 'රස', 'සංහිතාදි', 'තන්ත්ර', 'ශාස්ත්රයෙහි', 'වුත', 'කියන', 'ලද', 'විවිධං', 'නානාප්රකාර', 'වූ', 'සාරාර්ථය', 'උදුරා', 'ගෙණ', 'සකල', 'සත්වයන්ගේ', 'රොග', 'ව්යපසමන', 'සංඛ්යාත', 'සංග්රහව', 'පිණිස', 'මේ', 'සාරාර්ථ', 'සංග්රහ', 'නම්', 'ප්රකරණය', 'සර්ථාවබොධය', 'සඳහා', 'කියන්නෙමි']
Lemmas:   ['කාම', 'ලො', 'රූප', 'ලො', 'අරූපලෝ', 'යන්', 'ලොකත්රයට', 'ශ්රෙෂ්ඨ', 'ව', 'මුනීන්ද්ර', 'නම', 'ව', 'සම්', 'සම්බුද්ධ', 'වහන්සේ', 'ශ්ර', 'පාද', 'පද්ම', 'භ', 'ලොහ', 'කුලාචාරයෙන්', 'විනා', 'සකස', 'වඳ', 'සකල', 'ශාස්ත්රයෙහි', 'පාරප්රාප්ත', 'පණ්ඩිත', 'ව', 'භෛෂජ්යාර්ථයෙහි', 'දක්ෂ', 'ව', 'වෙද', 'විසින්', 'උතුම්', 'ව', 'රස', 'සංහිතාදි'

In [4]:
if 'century_lemmas' in locals():
    print(f"Saving to {OUTPUT_LEMMAS_FILE}...")
    with open(OUTPUT_LEMMAS_FILE, 'w', encoding='utf-8') as f:
        json.dump(century_lemmas, f, ensure_ascii=False, indent=2)
    print("Success.")
else:
    print("Skipping save (no data).")

Saving to Data/century_lemmas.json...
Success.
